In [ ]:
#segment only on map
import pandas as pd
import geopandas as gpd
from shapely import wkt
import folium


df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
df["geometry"] = df["geometry"].apply(wkt.loads)

edges = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:4326"
)

# boundary box so only load the edges necessary

north = 47.61758024981437
south = 47.613100153406556
east  = -122.32635157491734
west  = -122.33482914351927
edges_area = edges.cx[west:east, south:north]

# create map
m = folium.Map(
    location=[
        (north + south) / 2,
        (east + west) / 2
    ],
    zoom_start=17,
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors"
)

colors = [
    "#0aa519"
]

# add edges
for i, (_, row) in enumerate(edges_area.iterrows()):

    # geometry coordinates are lon,lat
    coords = [
        [lat, lon]
        for lon, lat in row.geometry.coords
    ]

    folium.PolyLine(
        coords,
        color=colors[i % len(colors)],
        weight=6,
        opacity=0.9
    ).add_to(m)
    
    # mark the segment midpoints so can differentiate them
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    folium.CircleMarker(
      location=[midpoint.y, midpoint.x],
      radius=5,          
      color="black",     
      weight=1,
      fill=True,
      fill_color="white",    
      fill_opacity=1,
      popup=folium.Popup(
          f"""
          OSM ID: {row['osm_id']}<br>
          u: {row['u']}<br>
          v: {row['v']}
          """,
          max_width=300
      )
    ).add_to(m)
 
    origin = [47.61671222562014, -122.33311978530625]
    destination = [47.61575302975626, -122.32800279032077]
   
   
    folium.Marker(
         origin,
         popup="Origin",
         icon=folium.Icon(color="green", icon="play")
    ).add_to(m)
 
   
  

# Display map
m.save("seattle_edges_map.html")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_61567/3862026418.py:7: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


In [ ]:
# code to show image ids along witrh segment
import folium
import geopandas as gpd
import pandas as pd
from shapely import wkt

# --- 1. Load Data & Bounding Box ---
df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
df["geometry"] = df["geometry"].apply(wkt.loads)

edges = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Boundary box definition
north = 47.61758024981437
south = 47.613100153406556
east = -122.32635157491734
west = -122.33482914351927

edges_area = edges.cx[west:east, south:north]

# Load images CSV
images_df = pd.read_csv("../Scores/2025_Survey_Images_Old_Scores.csv")
# Filter images to stay inside the same bounding box for performance and neatness
images_area = images_df[
    (images_df["Latitude"] >= south)
    & (images_df["Latitude"] <= north)
    & (images_df["Longitude"] >= west)
    & (images_df["Longitude"] <= east)
]

# --- 2. Create Folium Map ---
m = folium.Map(
    location=[(north + south) / 2, (east + west) / 2],
    zoom_start=17,
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
)

colors = ["#0aa519"]

# --- 3. Add Edges and Midpoint Markers ---
for i, (_, row) in enumerate(edges_area.iterrows()):
    # geometry coordinates are lon, lat
    coords = [[lat, lon] for lon, lat in row.geometry.coords]

    folium.PolyLine(
        coords, color=colors[i % len(colors)], weight=6, opacity=0.9
    ).add_to(m)

    # Mark segment midpoints
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    folium.CircleMarker(
        location=[midpoint.y, midpoint.x],
        radius=5,
        color="black",
        weight=1,
        fill=True,
        fill_color="white",
        fill_opacity=1,
        popup=folium.Popup(
            f"""
          OSM ID: {row['osm_id']}<br>
          u: {row['u']}<br>
          v: {row['v']}
          """,
            max_width=300,
        ),
    ).add_to(m)

# --- 4. Add Image Points (Small Red Circles with Hover Tooltips) ---
for _, img_row in images_area.iterrows():
    folium.CircleMarker(
        location=[img_row["Latitude"], img_row["Longitude"]],
        radius=3,  # Small circle
        color="red",
        weight=1,
        fill=True,
        fill_color="red",
        fill_opacity=0.8,
        tooltip=str(img_row["ImageID"]),  # Shows ImageID on hover
        popup=f"Image ID: {img_row['ImageID']}",  # Clickable popup details too
    ).add_to(m)

# --- 5. Add Origin and Destination Markers ---
origin = [47.61671222562014, -122.33311978530625]
destination = [47.61575302975626, -122.32800279032077]

folium.Marker(
    origin, popup="Origin", icon=folium.Icon(color="green", icon="play")
).add_to(m)

folium.Marker(
    destination,
    popup="Destination",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m)

# --- 6. Save Map ---
m.save("seattle_edges_map_with_images.html")
print("Map successfully saved as seattle_edges_map.html with image markers!")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_61567/2243503305.py:7: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


Map successfully saved as seattle_edges_map.html with image markers!


In [ ]:
import ast
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import Point

# 1. Load the data
paths_df = pd.read_csv("paths.csv")
edges_df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
images_df = pd.read_csv("../Scores/2025_Survey_Images_Old_Scores.csv")

# 2. Parse the segments_json column in paths_df
paths_df["segments"] = paths_df["segments_json"].apply(ast.literal_eval)

# Explode the segments so each row represents a single segment of a path
exploded_paths = paths_df.explode("segments").reset_index(drop=True)

# Extract 'u' and 'v' into separate columns
exploded_paths["u"] = exploded_paths["segments"].apply(lambda x: str(x["u"]))
exploded_paths["v"] = exploded_paths["segments"].apply(lambda x: str(x["v"]))

# 3. Clean and prepare edges GeoDataFrame
edges_df["u"] = edges_df["u"].astype(str)
edges_df["v"] = edges_df["v"].astype(str)

# Convert the WKT 'geometry' string to Shapely LineString geometries
edges_df["geometry"] = edges_df["geometry"].apply(wkt.loads)
edges_gdf = gpd.GeoDataFrame(edges_df, geometry="geometry", crs="EPSG:4326")

# 4. Merge paths with edge geometries using (u, v) pairs
path_segments = pd.merge(
    exploded_paths[["path_id", "u", "v"]],
    edges_gdf[["u", "v", "geometry"]],
    on=["u", "v"],
    how="inner",
)

# Convert to a GeoDataFrame (EPSG:4326 is WGS84 - lat/lon)
path_segments_gdf = gpd.GeoDataFrame(path_segments, geometry="geometry", crs="EPSG:4326")

# 5. Prepare images GeoDataFrame
images_df["geometry"] = images_df.apply(
    lambda row: Point(row["Longitude"], row["Latitude"]), axis=1
)
images_gdf = gpd.GeoDataFrame(images_df, geometry="geometry", crs="EPSG:4326")

# 6. Project both to a local projected coordinate system for accurate meter measurements
projected_crs = "EPSG:32610"
path_segments_proj = path_segments_gdf.to_crs(projected_crs)
images_proj = images_gdf.to_crs(projected_crs)

# 7. Buffer the path segments by 20 meters
path_segments_proj["buffer"] = path_segments_proj.geometry.buffer(20)

# Create a separate GeoDataFrame specifically for the buffered zones
buffered_gdf = gpd.GeoDataFrame(
    path_segments_proj[["path_id", "u", "v"]],
    geometry=path_segments_proj["buffer"],
    crs=projected_crs,
)

# 8. Spatial join to find images within 20m of the path segments
result = gpd.sjoin(images_proj, buffered_gdf, how="inner", predicate="within")

# Drop duplicate images if multiple segments of the same path cover the same image
unique_matches = result.drop_duplicates(subset=["path_id", "ImageID"])

# 9. Group by path_id and aggregate images into lists per path
final_output = (
    unique_matches.groupby("path_id")
    .agg(
        image_list=("ImageID", list),
        image_count=("ImageID", "count"),
        all_image_details=(
            "ImageID",
            lambda x: list(
                unique_matches.loc[
                    unique_matches["ImageID"].isin(x),
                    [
                        "ImageID",
                        "Latitude",
                        "Longitude",
                        "Walkability Rating",
                        "OpenAI Score",
                    ],
                ].to_dict(orient="records")
            ),
        ),
    )
    .reset_index()
)

# Save results
final_output.to_csv("path_to_images_summary.csv", index=False)
print(f"Successfully processed {len(final_output)} paths with their image lists!")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_61567/28413578.py:9: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  edges_df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


Successfully processed 4 paths with their image lists!
